# Semana 12 · Visualización e interpretación de resultados  
**Proyecto Productivo IB — Instituto Continental · Equipo AgroData IC (Grupo 03)**  
**Proyecto:** Sistema de alerta temprana basado en grados-día para la arañita roja (*Oligonychus punicae*) en palto Hass · Fundo Loma Verde S.A.C. (Supe, Barranca)

Este notebook sigue el patrón del notebook docente de Piedra Lisa (cargar → verificar → calcular la métrica → graficar → interpretar → exportar), pero con **nuestros datos reales** (`aranita_roja_bd_v1.sqlite`, Semanas 6 a 8), no con datos sintéticos.

**Qué produce:** tres visualizaciones que pide la docente para este proyecto (serie temporal, dispersión con temperatura y dispersión con humedad), una cuarta complementaria (mapa de calor módulo × mes) que conecta con la decisión del semáforo, la tabla de interpretación y las tablas de respaldo.

**Cómo usarlo en Google Colab:** ejecute las celdas en orden. La primera le pedirá subir `aranita_roja_bd_v1.sqlite`. Todo lo demás se calcula desde esa base; no se generan ni se sustituyen datos.

> **Regla de la docente:** no se acepta un gráfico sin poder explicar qué variable representa y cómo se obtuvo su valor. Por eso cada gráfico tiene debajo su verificación en código (`assert`) y su ficha.

## Bloque 1 · Carga y verificación de la base  
**Objetivo:** abrir la base SQLite y confirmar que es la versión v1 que validamos en la Semana 8 (183 días de clima y 1,494 evaluaciones).

In [ ]:
import os, sqlite3, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import matplotlib.patheffects as pe
from scipy.stats import spearmanr
from IPython.display import display

# --- Cargar la base: en Colab se sube desde el computador; fuera de Colab se usa la ruta local.
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    subidos = files.upload()                      # seleccione aranita_roja_bd_v1.sqlite
    if not subidos:
        raise ValueError('No se seleccionó ningún archivo.')
    RUTA_BD = next(iter(subidos))
else:
    RUTA_BD = 'aranita_roja_bd_v1.sqlite'         # misma carpeta del notebook

if not RUTA_BD.lower().endswith('.sqlite') and not RUTA_BD.lower().endswith('.db'):
    raise ValueError(f'El archivo no parece una base SQLite: {RUTA_BD}')

con = sqlite3.connect(RUTA_BD)
print('Base abierta:', RUTA_BD)
print('Tablas:', [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

In [ ]:
# Verificación: la base debe ser la v1 de la Semana 8.
n_clima = con.execute('SELECT COUNT(*) FROM clima_diario').fetchone()[0]
n_eval  = con.execute('SELECT COUNT(*) FROM evaluacion_fitosanitaria').fetchone()[0]
f_min, f_max = con.execute('SELECT MIN(fecha), MAX(fecha) FROM clima_diario').fetchone()
print('Días de clima:', n_clima, '| periodo:', f_min, 'a', f_max)
print('Evaluaciones fitosanitarias:', n_eval)
print('Evaluaciones por estado de calidad:',
      dict(con.execute('SELECT estado_calidad_evaluacion, COUNT(*) FROM evaluacion_fitosanitaria GROUP BY 1').fetchall()))
assert n_clima == 183 and n_eval == 1494, 'No es la base v1 de la Semana 8'
assert (f_min, f_max) == ('2026-03-01', '2026-08-30')
print('Verificación de la base: CORRECTA')

## Bloque 2 · Dataset analítico v1 (consulta con JOIN)  
**Unidad de análisis:** una **evaluación fitosanitaria** = una fila para una combinación de *fecha + módulo + válvula + turno*, unida al clima de su día.

**¿Por qué solo `ACEPTADO`?** En la Semana 7 la validación dejó 1,462 evaluaciones `ACEPTADO` y 32 `OBSERVADO` (30 duplicados lógicos corregidos y 2 valores atípicos de grado de postura). Para graficar usamos solo las aceptadas, así cada punto viene de un registro que pasó todas las reglas. Los `OBSERVADO` no se eliminaron de la base: simplemente no entran a este análisis.

**Variable elegida como indicador:** `aranita_hojas_infestadas` (% de hojas infestadas). No usamos `aranita_infestacion_postura` (% de postura, es decir huevos) porque vale 0 en el 90.1 % de las evaluaciones y casi no discrimina.

In [ ]:
SQL_ANALITICO = '''
SELECT e.evaluacion_id,
       e.fecha,
       m.modulo,
       t.turno,
       e.valvula,
       e.plantas_evaluadas,
       e.aranita_hojas_infestadas   AS hojas_inf_pct,
       e.aranita_infestacion_postura AS postura_pct,
       c.temperatura_oc,
       c.humedad_relativa,
       c.estado_calidad_clima
FROM evaluacion_fitosanitaria e
JOIN clima_diario c ON c.fecha = e.fecha
JOIN dim_modulo   m ON m.modulo_id = e.modulo_id
JOIN dim_turno    t ON t.turno_id  = e.turno_id
WHERE e.estado_calidad_evaluacion = 'ACEPTADO'
ORDER BY e.fecha, m.modulo, e.valvula
'''
df = pd.read_sql(SQL_ANALITICO, con)
df['fecha'] = pd.to_datetime(df['fecha'])
df['mes'] = df['fecha'].dt.month                      # 3 = marzo ... 8 = agosto
display(df.head())
print('Filas:', len(df), '| fechas distintas:', df['fecha'].nunique(),
      '| periodo:', df['fecha'].min().date(), 'a', df['fecha'].max().date())

In [ ]:
# Controles de calidad del dataset analítico (si algo falla, el notebook se detiene).
assert len(df) == 1462, 'Se esperan 1,462 evaluaciones ACEPTADO'
assert df['fecha'].nunique() == 50
assert df.duplicated(['fecha','modulo','valvula','turno']).sum() == 0, 'Hay evaluaciones repetidas'
assert df[['hojas_inf_pct','temperatura_oc','humedad_relativa']].isna().sum().sum() == 0
assert df['hojas_inf_pct'].between(0, 100).all(), '% de hojas fuera de 0-100'
assert df['humedad_relativa'].between(0, 100).all()
assert set(df['modulo']) == {1, 2, 3, 4, 5, 6}
print('Postura = 0 en', round((df['postura_pct'] == 0).mean() * 100, 1), '% de las evaluaciones')
assert round((df['postura_pct'] == 0).mean() * 100, 1) == 90.1
print('Verificación del dataset analítico: CORRECTA')

In [ ]:
# Carpeta de entrega con la estructura que pide la docente.
RAIZ = 'Semana_12_Visualizacion'
for sub in ['dataset_y_fuentes', 'graficos', 'analisis', 'matriz_interpretacion']:
    os.makedirs(os.path.join(RAIZ, sub), exist_ok=True)

df.assign(fecha=df['fecha'].dt.strftime('%Y-%m-%d')).to_csv(
    os.path.join(RAIZ, 'dataset_y_fuentes', 'aranita_roja_analitico_v1.csv'), index=False, encoding='utf-8-sig')

# Estilo: un solo color de acento, ejes limpios, cuadrícula suave.
AZUL, GRIS, GRIS_CLARO = '#0D5192', '#5B6573', '#B8C0CC'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#9CA3AF', 'axes.grid': True, 'grid.color': '#E5E7EB',
                     'grid.linewidth': 0.6, 'axes.axisbelow': True, 'figure.dpi': 110})
MESES = {3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Ago'}
FUENTE = 'Fuente: Fundo Loma Verde S.A.C., estación meteorológica y monitoreo fitosanitario · aranita_roja_bd_v1 · evaluaciones ACEPTADO'
print('Carpetas listas en:', os.path.abspath(RAIZ))

## Figura 1 · Serie temporal: ¿cómo evoluciona la infestación?  
**Pregunta analítica:** ¿cómo evoluciona el % de hojas infestadas entre marzo y agosto de 2026?  
**Por qué líneas:** la pregunta es cómo cambia una variable en el tiempo. Las condiciones de la docente para un gráfico de tiempo: fechas bien interpretadas, intervalos comparables, métrica calculada de forma consistente y periodos incompletos identificados. Aquí los 6 meses tienen evaluaciones desde el inicio hasta el final de cada mes (marzo 2 al 27 … agosto 1 al 28), así que son comparables.  
**Métrica:** promedio de **todas las evaluaciones aceptadas del mes** (no el promedio de promedios).

In [ ]:
# --- 1) Métrica: promedio y mediana mensual (sobre evaluaciones) y promedio por fecha (para los puntos grises).
mensual = (df.groupby('mes')['hojas_inf_pct']
             .agg(n='size', promedio='mean', mediana='median').round(2))
display(mensual)

por_fecha = (df.groupby('fecha')
               .agg(n=('evaluacion_id', 'size'), hoja=('hojas_inf_pct', 'mean'),
                    T=('temperatura_oc', 'first'), HR=('humedad_relativa', 'first'),
                    clima_ok=('estado_calidad_clima', 'first'))
               .reset_index())

# Posición de cada punto mensual: centro de las fechas de evaluación de ese mes.
centro_mes = df.groupby('mes')['fecha'].agg(lambda s: s.min() + (s.max() - s.min()) / 2)

# --- 2) Verificación antes de graficar.
assert mensual['n'].sum() == len(df) == 1462
assert mensual['promedio'].round(2).tolist() == [3.01, 3.72, 7.26, 12.59, 11.16, 16.45]
assert mensual['mediana'].round(2).tolist() == [0.0, 0.0, 4.68, 3.12, 5.35, 6.25]
assert round(mensual.loc[8, 'promedio'] / mensual.loc[3, 'promedio'], 1) == 5.5    # 'unas 5.5 veces' de la ficha
print('Verificación de la métrica mensual: CORRECTA')

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.9))
# Puntos grises: promedio de cada fecha de evaluación (tamaño = n.º de evaluaciones ese día).
ax.scatter(por_fecha['fecha'], por_fecha['hoja'], s=12 + por_fecha['n'] * 1.6, color=GRIS_CLARO,
           edgecolor='white', linewidth=0.6, zorder=2, label='Promedio de cada fecha de evaluación (tamaño = n.º de evaluaciones)')
# Línea azul: promedio mensual.
ax.plot(centro_mes.values, mensual['promedio'].values, color=AZUL, linewidth=2.2, marker='o', markersize=7,
        markeredgecolor='white', markeredgewidth=1.2, zorder=4, label='Promedio mensual (todas las evaluaciones del mes)')
for x, v in zip(centro_mes.values, mensual['promedio'].values):
    ax.annotate(f'{v:.1f} %', (x, v), textcoords='offset points', xytext=(0, 9), ha='center',
                fontsize=9.5, fontweight='bold', color=AZUL,
                path_effects=[pe.withStroke(linewidth=3, foreground='white')])
ax.set_title('Infestación por arañita roja: de 3.0 % en marzo a 16.4 % en agosto', loc='left', fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.015, 'Promedio mensual del % de hojas infestadas · Fundo Loma Verde, 2 mar - 28 ago 2026 · n = 1,462 evaluaciones en 50 fechas',
        transform=ax.transAxes, fontsize=8.8, color=GRIS, va='bottom')
ax.set_xlabel('Mes de 2026')
ax.set_ylabel('Hojas infestadas (%)')
ax.set_ylim(0, 46)
ax.set_xlim(pd.Timestamp('2026-02-24'), pd.Timestamp('2026-09-05'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',7:'Jul',8:'Ago',9:'Sep'}[mdates.num2date(x).month]))
ax.legend(loc='upper left', frameon=False, fontsize=8.6)
fig.text(0.01, 0.005, FUENTE, fontsize=7, color=GRIS)
fig.tight_layout(rect=(0, 0.02, 1, 1))
fig.savefig(os.path.join(RAIZ, 'graficos', 'figura_01_serie_temporal_infestacion.png'), dpi=170, bbox_inches='tight')
plt.show()

### Ficha · Figura 1: Evolución mensual del % de hojas infestadas (serie temporal)

| Campo | Contenido |
|---|---|
| **Pregunta analítica** | ¿Cómo evoluciona el % de hojas infestadas por arañita roja entre marzo y agosto de 2026? |
| **Variables (campos reales)** | evaluacion_fitosanitaria.fecha; aranita_hojas_infestadas (% de hojas infestadas); filtro estado_calidad_evaluacion = ACEPTADO |
| **Unidad de análisis** | Una evaluación fitosanitaria (fecha + módulo + válvula + turno). n = 1,462 evaluaciones en 50 fechas, agrupadas en 6 meses. |
| **Métrica / operación estadística** | Promedio mensual del % de hojas infestadas (y mediana como control). |
| **Visualización y justificación** | Líneas: la pregunta es cómo cambia una variable en el tiempo. Puntos grises = promedio de cada fecha de evaluación. |
| **Resultado observado (cifras)** | Promedio mensual: mar 3.01 %, abr 3.72 %, may 7.26 %, jun 12.59 %, jul 11.16 %, ago 16.45 %. Mediana mensual: 0, 0, 4.68, 3.12, 5.35 y 6.25 %. |
| **Interpretación** | En los registros disponibles, el promedio de agosto (16.45 %) es unas 5.5 veces el de marzo (3.01 %). La mayor subida ocurre entre abril y junio (3.72 % a 12.59 %); después se mantiene entre 11 % y 16 %. |
| **Limitación** | La distribución es asimétrica: la mediana de agosto (6.25 %) es mucho menor que su promedio, porque pocas válvulas con valores muy altos lo elevan. Hay un solo ciclo de 6 meses (no se puede hablar de estacionalidad), las fechas de evaluación son irregulares (1 a 10 días entre una y otra) y el gráfico no identifica la causa del aumento. |
| **Decisión que podría evaluarse** | Revisar con el Ing. Kiyoshi desde qué nivel de infestación corresponde activar una intervención y si el monitoreo se refuerza desde mayo, cuando el promedio empieza a subir. |
| **Archivo y versión** | figura_01_serie_temporal_infestacion.png · S12_AgroDataIC_v1.ipynb · aranita_roja_bd_v1.sqlite (v1) · aranita_roja_analitico_v1.csv |

## Figuras 2 y 3 · Dispersión: ¿se asocian temperatura y humedad con la infestación?  
**Preguntas analíticas:** ¿existe asociación entre la temperatura media del día y el % de hojas infestadas? ¿y con la humedad relativa media?  
**Por qué dispersión:** son dos variables cuantitativas y queremos ver si cambian de forma asociada.

**Emparejamiento (lo que más cuida la docente):** el clima es diario y hay muchas evaluaciones por día, así que cada punto es una **fecha de evaluación**: el promedio del % de hojas infestadas de ese día emparejado con la temperatura (o humedad) de **ese mismo día**. Si graficáramos las 1,462 evaluaciones, los puntos formarían columnas (todas las evaluaciones del día comparten el mismo clima) y parecería que hay más datos independientes de los que realmente existen. Usamos solo días de clima `ACEPTADO` (47 de las 50 fechas).

**Advertencia:** una asociación (o su ausencia) en un gráfico de dispersión no demuestra causa.

In [ ]:
# Fechas con clima validado (ACEPTADO): 47 de 50.
base = por_fecha[por_fecha['clima_ok'] == 'ACEPTADO'].copy()
print('Fechas de evaluación:', len(por_fecha), '| con clima ACEPTADO:', len(base))
print('Evaluaciones por fecha (en esas 47): mín =', base['n'].min(), '| máx =', base['n'].max())
print('Temperatura media: de', base['T'].min(), 'a', base['T'].max(), '°C | amplitud =', round(base['T'].max() - base['T'].min(), 1), '°C')
print('Humedad relativa media: de', base['HR'].min(), 'a', base['HR'].max(), '%')

def spear(d, x):
    r = spearmanr(d[x], d['hoja'])
    return round(float(r[0]), 3), round(float(r[1]), 3), len(d)

rho_T, p_T, n_T = spear(base, 'T')
rho_H, p_H, n_H = spear(base, 'HR')
print('Temperatura: rho =', rho_T, '| p =', p_T, '| n =', n_T)
print('Humedad:     rho =', rho_H, '| p =', p_H, '| n =', n_H)

assert len(base) == 47 and (base['n'].min(), base['n'].max()) == (2, 79)
assert (base['n'] < 10).sum() == 8                  # 'hay 8 fechas con menos de 10 evaluaciones'
assert abs(rho_T - 0.055) < 0.002 and abs(p_T - 0.713) < 0.002
assert abs(rho_H + 0.113) < 0.002 and abs(p_H - 0.450) < 0.002
assert (round(base['T'].min(), 1), round(base['T'].max(), 1)) == (20.4, 23.2)
assert (round(base['HR'].min(), 1), round(base['HR'].max(), 1)) == (74.2, 89.0)
print('Verificación de las correlaciones: CORRECTA')

In [ ]:
def dispersion(x, xlabel, titulo, subtitulo, rho, p, nobs, xlim, archivo):
    fig, ax = plt.subplots(figsize=(8.2, 4.9))
    ax.scatter(base[x], base['hoja'], s=16 + base['n'] * 3.2, color=AZUL, alpha=0.72,
               edgecolor='white', linewidth=0.8, zorder=3)
    ax.set_title(titulo, loc='left', fontsize=12.5, fontweight='bold', pad=26)
    ax.text(0, 1.015, subtitulo, transform=ax.transAxes, fontsize=8.8, color=GRIS, va='bottom')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Hojas infestadas (%), promedio de la fecha')
    ax.set_xlim(*xlim)
    ax.set_ylim(0, 46)
    ax.text(0.02, 0.96, f'rho de Spearman = {rho:.2f}  (p = {p:.2f})\nn = {nobs} fechas',
            transform=ax.transAxes, va='top', fontsize=9.5, color='#1F2937',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#D1D5DB'))
    fig.text(0.01, 0.005, 'Tamaño del punto = n.º de evaluaciones de esa fecha (de 2 a 79). ' + FUENTE, fontsize=6.8, color=GRIS)
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    fig.savefig(os.path.join(RAIZ, 'graficos', archivo), dpi=170, bbox_inches='tight')
    plt.show()

dispersion('T', 'Temperatura media del día (°C)',
           'La temperatura no explica la infestación: casi no varía entre fechas',
           'Cada punto = una fecha de evaluación (mar-ago 2026) · temperatura media entre 20.4 y 23.2 °C',
           rho_T, p_T, n_T, (19.5, 24.0), 'figura_02_dispersion_temperatura_infestacion.png')

### Ficha · Figura 2: Temperatura media del día vs. % de hojas infestadas (dispersión)

| Campo | Contenido |
|---|---|
| **Pregunta analítica** | ¿Existe asociación entre la temperatura media del día y el % de hojas infestadas? |
| **Variables (campos reales)** | clima_diario.temperatura_oc (°C, media diaria) y aranita_hojas_infestadas (%), emparejadas por fecha; solo evaluaciones y días de clima ACEPTADO |
| **Unidad de análisis** | Una fecha de evaluación (cada punto = promedio del % de hojas infestadas de todas las evaluaciones de ese día, con el clima de ese día). n = 47 fechas. |
| **Métrica / operación estadística** | Promedio por fecha y correlación de Spearman (rho). |
| **Visualización y justificación** | Dispersión: son dos variables cuantitativas y se quiere ver si cambian de forma asociada. El tamaño del punto indica cuántas evaluaciones tiene la fecha. |
| **Resultado observado (cifras)** | rho de Spearman = 0.06 (p = 0.71), n = 47. La temperatura media de esas fechas va de 20.4 °C a 23.2 °C. |
| **Interpretación** | En estos datos no se observa asociación entre la temperatura media del día y la infestación promedio. El rango térmico es tan estrecho (unos 2.7 °C) que este conjunto casi no permite evaluar el efecto de la temperatura. |
| **Limitación** | Rango térmico muy estrecho; el emparejamiento es del mismo día (la plaga puede responder con retraso); con el promedio de los 7 días que terminan en la fecha, rho = -0.21 (p = 0.17, n = 45), tampoco una asociación positiva; hay 8 fechas con menos de 10 evaluaciones y promedios inestables (solo con n >= 10: rho = -0.06); una sola estación y un solo ciclo. Una correlación no demuestra causa. |
| **Decisión que podría evaluarse** | No usar la temperatura media diaria como único criterio del semáforo. Mantener el grado-día como contexto térmico (Tbase y K siguen siendo supuestos de literatura sin validar) y apoyar la alerta en la infestación observada en campo. |
| **Archivo y versión** | figura_02_dispersion_temperatura_infestacion.png · S12_AgroDataIC_v1.ipynb · aranita_roja_bd_v1.sqlite (v1) · aranita_roja_analitico_v1.csv |

In [ ]:
dispersion('HR', 'Humedad relativa media del día (%)',
           'La humedad relativa tampoco muestra una asociación clara con la infestación',
           'Cada punto = una fecha de evaluación (mar-ago 2026) · humedad media entre 74.2 y 89.0 %',
           rho_H, p_H, n_H, (72, 91), 'figura_03_dispersion_humedad_infestacion.png')

### Ficha · Figura 3: Humedad relativa media del día vs. % de hojas infestadas (dispersión)

| Campo | Contenido |
|---|---|
| **Pregunta analítica** | ¿Existe asociación entre la humedad relativa media del día y el % de hojas infestadas? |
| **Variables (campos reales)** | clima_diario.humedad_relativa (%, media diaria) y aranita_hojas_infestadas (%), emparejadas por fecha; solo evaluaciones y días de clima ACEPTADO |
| **Unidad de análisis** | Una fecha de evaluación (cada punto = promedio del % de hojas infestadas de ese día, con la humedad de ese día). n = 47 fechas. |
| **Métrica / operación estadística** | Promedio por fecha y correlación de Spearman (rho). |
| **Visualización y justificación** | Dispersión por la misma razón que la Figura 2: dos variables cuantitativas emparejadas por fecha. |
| **Resultado observado (cifras)** | rho de Spearman = -0.11 (p = 0.45), n = 47. La humedad relativa media de esas fechas va de 74.2 % a 89.0 %. |
| **Interpretación** | En estos datos no se detecta una asociación entre la humedad relativa del día y la infestación promedio: el signo es ligeramente negativo, pero es débil y no es estadísticamente distinguible de cero. |
| **Limitación** | Mismas limitaciones de la Figura 2: mismo día, una sola estación, un ciclo y fechas con pocas evaluaciones. Con el promedio de 7 días, rho = 0.02 (p = 0.89, n = 45); solo con n >= 10, rho = 0.10. Una correlación no demuestra causa. |
| **Decisión que podría evaluarse** | No incluir la humedad relativa como variable de alerta por ahora. Conservarla en el dataset para revisar la relación cuando existan más campañas. |
| **Archivo y versión** | figura_03_dispersion_humedad_infestacion.png · S12_AgroDataIC_v1.ipynb · aranita_roja_bd_v1.sqlite (v1) · aranita_roja_analitico_v1.csv |

### Prueba de sensibilidad: ¿cambia la conclusión si el emparejamiento o el filtro son distintos?
La docente advirtió que emparejar mediciones que no corresponden en el tiempo puede crear asociaciones falsas o esconder las reales. Por eso repetimos la correlación con tres variantes razonables. Si en todas el resultado es el mismo (ninguna asociación clara), la conclusión es más sólida.

In [ ]:
clima = pd.read_sql('SELECT fecha, temperatura_oc AS T, humedad_relativa AS HR FROM clima_diario ORDER BY fecha', con)
clima['fecha'] = pd.to_datetime(clima['fecha'])
clima['T7'] = clima['T'].rolling(7).mean()         # promedio de los 7 días que terminan en la fecha
clima['HR7'] = clima['HR'].rolling(7).mean()
b7 = base.merge(clima[['fecha', 'T7', 'HR7']], on='fecha').dropna(subset=['T7', 'HR7'])

filas = []
variantes = [('Base: mismo día, clima ACEPTADO', base, 'T', 'HR'),
             ('Todas las fechas (incluye clima OBSERVADO)', por_fecha, 'T', 'HR'),
             ('Solo fechas con 10 o más evaluaciones', base[base['n'] >= 10], 'T', 'HR'),
             ('Promedio de 7 días que terminan en la fecha', b7, 'T7', 'HR7')]
for nombre, d, xt, xh in variantes:
    rt, pt, nt = spear(d, xt)
    rh, ph, nh = spear(d, xh)
    filas.append({'variante': nombre, 'n_fechas': nt, 'rho_temperatura': rt, 'p_temperatura': pt, 'rho_humedad': rh, 'p_humedad': ph})
sens = pd.DataFrame(filas)
display(sens)
sens.to_csv(os.path.join(RAIZ, 'analisis', 'tabla_sensibilidad_correlaciones.csv'), index=False, encoding='utf-8-sig')

assert sens['n_fechas'].tolist() == [47, 50, 39, 45]
assert abs(sens.loc[3, 'rho_temperatura'] + 0.210) < 0.003 and abs(sens.loc[3, 'rho_humedad'] - 0.022) < 0.003
assert abs(sens.loc[2, 'rho_temperatura'] + 0.057) < 0.003 and abs(sens.loc[2, 'rho_humedad'] - 0.103) < 0.003
assert (sens[['p_temperatura', 'p_humedad']] > 0.05).all().all(), 'Alguna variante dio asociación significativa: revisar interpretación'
print('Ninguna variante muestra asociación significativa (todos los p > 0.05): CORRECTA la interpretación de las Figuras 2 y 3')

## Figura 4 (complementaria) · Mapa de calor: ¿en qué módulos y meses se concentra la infestación?  
**Pregunta analítica:** ¿en qué módulos y meses se concentra el mayor % de hojas infestadas?  
**Por qué matriz de calor:** combinamos dos categorías (módulo y mes) y una medida numérica. Es la figura que más se conecta con la decisión del proyecto (a qué módulo mirar primero y cuándo).  
**Control:** las celdas con menos de 5 evaluaciones no se pintan, porque su promedio no es confiable (el módulo 1 solo tiene 9 evaluaciones válidas en total). Cada celda muestra el valor y su n para que el color no se lea solo.

In [ ]:
pv = df.pivot_table(index='modulo', columns='mes', values='hojas_inf_pct', aggfunc='mean')
pn = df.pivot_table(index='modulo', columns='mes', values='hojas_inf_pct', aggfunc='size').fillna(0).astype(int)
pv_ok = pv.where(pn >= 5)                      # celdas con n >= 5

display(pv.round(1)); display(pn)
assert pn.to_numpy().sum() == 1462
assert round(pv.loc[6, 8], 1) == 27.4 and round(pv.loc[6, 5], 1) == 0.6
assert [round(v, 1) for v in pv.loc[4, [6, 7, 8]]] == [15.9, 19.4, 19.7]
assert [round(v, 1) for v in pv.loc[5, [6, 7, 8]]] == [18.5, 17.4, 23.7]
assert round(pv_ok.loc[[2, 3]].max().max(), 1) == 12.9
assert (pn.loc[1].sum()) == 9 and (pn.loc[2:] .to_numpy().min(), pn.loc[2:].to_numpy().max()) == (17, 89)
print('Verificación de la matriz módulo x mes: CORRECTA')

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.9))
datos = np.ma.masked_invalid(pv_ok.to_numpy())
cmap = plt.get_cmap('Blues').copy(); cmap.set_bad('#F1F3F5')
im = ax.imshow(datos, cmap=cmap, vmin=0, vmax=30, aspect='auto')
ax.set_xticks(range(len(pv.columns))); ax.set_xticklabels([MESES[m] for m in pv.columns])
ax.set_yticks(range(len(pv.index))); ax.set_yticklabels([f'Módulo {m}' for m in pv.index])
ax.grid(False)
for i, mod in enumerate(pv.index):
    for j, mes in enumerate(pv.columns):
        n = pn.loc[mod, mes]
        if n >= 5:
            v = pv.loc[mod, mes]
            ax.text(j, i - 0.1, f'{v:.1f}', ha='center', va='center', fontsize=10.5, fontweight='bold',
                    color='white' if v >= 15 else '#111827')
            ax.text(j, i + 0.27, f'n={n}', ha='center', va='center', fontsize=7.4,
                    color='white' if v >= 15 else GRIS)
        else:
            ax.text(j, i, f'n={n}\n(no se pinta)', ha='center', va='center', fontsize=7, color='#8A93A0')
for s in ax.spines.values(): s.set_visible(False)
ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, label='Hojas infestadas (%), promedio de la celda', pad=0.02)
cb.outline.set_visible(False)
ax.set_title('Los módulos 4 y 5 superan 15 % desde junio; el módulo 6, en agosto', loc='left', fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.015, 'Promedio del % de hojas infestadas por módulo y mes · mar-ago 2026 · n = 1,462 evaluaciones',
        transform=ax.transAxes, fontsize=8.8, color=GRIS, va='bottom')
fig.text(0.01, 0.005, FUENTE + ' · celdas con n < 5 sin pintar', fontsize=6.8, color=GRIS)
fig.tight_layout(rect=(0, 0.02, 1, 1))
fig.savefig(os.path.join(RAIZ, 'graficos', 'figura_04_mapa_calor_modulo_mes.png'), dpi=170, bbox_inches='tight')
plt.show()

### Ficha · Figura 4 (complementaria): Mapa de calor módulo × mes del % promedio de hojas infestadas

| Campo | Contenido |
|---|---|
| **Pregunta analítica** | ¿En qué módulos y meses se concentra el mayor % de hojas infestadas? |
| **Variables (campos reales)** | dim_modulo.modulo; evaluacion_fitosanitaria.fecha (mes) y aranita_hojas_infestadas (%); filtro ACEPTADO |
| **Unidad de análisis** | Una evaluación fitosanitaria; n = 1,462, agrupadas por módulo × mes (las celdas con menos de 5 evaluaciones no se pintan). |
| **Métrica / operación estadística** | Promedio del % de hojas infestadas por celda módulo × mes. |
| **Visualización y justificación** | Matriz de calor: se comparan dos dimensiones categóricas (módulo y mes) con una medida numérica. Cada celda muestra el valor y su n. |
| **Resultado observado (cifras)** | El módulo 6 pasa de 0.6 % (mayo) a 27.4 % (agosto). Los módulos 4 y 5 superan 15 % de junio a agosto (módulo 4: 15.9, 19.4 y 19.7 %; módulo 5: 18.5, 17.4 y 23.7 %). Los módulos 2 y 3 no pasan de 12.9 % en ningún mes. |
| **Interpretación** | En los registros disponibles, la infestación se concentra en los módulos 4 y 5 desde junio y en el módulo 6 en agosto, mientras que los módulos 2 y 3 se mantienen por debajo de 13 %. Que el módulo 6 solo se eleve en agosto sugiere un aumento más tardío, no una infestación constante. |
| **Limitación** | El módulo 1 solo tiene 9 evaluaciones válidas y no se pinta. El n por celda va de 17 a 89. No se dispone de información sobre aplicaciones de control ni sobre diferencias de edad o variedad entre módulos; una baja puede deberse a un tratamiento. El promedio está influido por pocas válvulas con valores altos. |
| **Decisión que podría evaluarse** | Priorizar la revisión en campo de los módulos 4, 5 y 6 y definir con el Ing. Kiyoshi los umbrales del semáforo por módulo antes de fijarlos. |
| **Archivo y versión** | figura_04_mapa_calor_modulo_mes.png · S12_AgroDataIC_v1.ipynb · aranita_roja_bd_v1.sqlite (v1) · aranita_roja_analitico_v1.csv |

## Exportación · tablas de respaldo y matriz de interpretación
La docente pide guardar el notebook, las figuras y una **matriz de interpretación** (una fila por visualización: pregunta, variables, métrica, resultado, interpretación, limitación y decisión), más la trazabilidad de los datos.

In [ ]:
FICHAS = {
 "Figura 1": {
  "titulo": "Evolución mensual del % de hojas infestadas (serie temporal)",
  "pregunta": "¿Cómo evoluciona el % de hojas infestadas por arañita roja entre marzo y agosto de 2026?",
  "variables": "evaluacion_fitosanitaria.fecha; aranita_hojas_infestadas (% de hojas infestadas); filtro estado_calidad_evaluacion = ACEPTADO",
  "unidad": "Una evaluación fitosanitaria (fecha + módulo + válvula + turno). n = 1,462 evaluaciones en 50 fechas, agrupadas en 6 meses.",
  "metrica": "Promedio mensual del % de hojas infestadas (y mediana como control).",
  "visualizacion": "Líneas: la pregunta es cómo cambia una variable en el tiempo. Puntos grises = promedio de cada fecha de evaluación.",
  "resultado": "Promedio mensual: mar 3.01 %, abr 3.72 %, may 7.26 %, jun 12.59 %, jul 11.16 %, ago 16.45 %. Mediana mensual: 0, 0, 4.68, 3.12, 5.35 y 6.25 %.",
  "interpretacion": "En los registros disponibles, el promedio de agosto (16.45 %) es unas 5.5 veces el de marzo (3.01 %). La mayor subida ocurre entre abril y junio (3.72 % a 12.59 %); después se mantiene entre 11 % y 16 %.",
  "limitacion": "La distribución es asimétrica: la mediana de agosto (6.25 %) es mucho menor que su promedio, porque pocas válvulas con valores muy altos lo elevan. Hay un solo ciclo de 6 meses (no se puede hablar de estacionalidad), las fechas de evaluación son irregulares (1 a 10 días entre una y otra) y el gráfico no identifica la causa del aumento.",
  "decision": "Revisar con el Ing. Kiyoshi desde qué nivel de infestación corresponde activar una intervención y si el monitoreo se refuerza desde mayo, cuando el promedio empieza a subir."
 },
 "Figura 2": {
  "titulo": "Temperatura media del día vs. % de hojas infestadas (dispersión)",
  "pregunta": "¿Existe asociación entre la temperatura media del día y el % de hojas infestadas?",
  "variables": "clima_diario.temperatura_oc (°C, media diaria) y aranita_hojas_infestadas (%), emparejadas por fecha; solo evaluaciones y días de clima ACEPTADO",
  "unidad": "Una fecha de evaluación (cada punto = promedio del % de hojas infestadas de todas las evaluaciones de ese día, con el clima de ese día). n = 47 fechas.",
  "metrica": "Promedio por fecha y correlación de Spearman (rho).",
  "visualizacion": "Dispersión: son dos variables cuantitativas y se quiere ver si cambian de forma asociada. El tamaño del punto indica cuántas evaluaciones tiene la fecha.",
  "resultado": "rho de Spearman = 0.06 (p = 0.71), n = 47. La temperatura media de esas fechas va de 20.4 °C a 23.2 °C.",
  "interpretacion": "En estos datos no se observa asociación entre la temperatura media del día y la infestación promedio. El rango térmico es tan estrecho (unos 2.7 °C) que este conjunto casi no permite evaluar el efecto de la temperatura.",
  "limitacion": "Rango térmico muy estrecho; el emparejamiento es del mismo día (la plaga puede responder con retraso); con el promedio de los 7 días que terminan en la fecha, rho = -0.21 (p = 0.17, n = 45), tampoco una asociación positiva; hay 8 fechas con menos de 10 evaluaciones y promedios inestables (solo con n >= 10: rho = -0.06); una sola estación y un solo ciclo. Una correlación no demuestra causa.",
  "decision": "No usar la temperatura media diaria como único criterio del semáforo. Mantener el grado-día como contexto térmico (Tbase y K siguen siendo supuestos de literatura sin validar) y apoyar la alerta en la infestación observada en campo."
 },
 "Figura 3": {
  "titulo": "Humedad relativa media del día vs. % de hojas infestadas (dispersión)",
  "pregunta": "¿Existe asociación entre la humedad relativa media del día y el % de hojas infestadas?",
  "variables": "clima_diario.humedad_relativa (%, media diaria) y aranita_hojas_infestadas (%), emparejadas por fecha; solo evaluaciones y días de clima ACEPTADO",
  "unidad": "Una fecha de evaluación (cada punto = promedio del % de hojas infestadas de ese día, con la humedad de ese día). n = 47 fechas.",
  "metrica": "Promedio por fecha y correlación de Spearman (rho).",
  "visualizacion": "Dispersión por la misma razón que la Figura 2: dos variables cuantitativas emparejadas por fecha.",
  "resultado": "rho de Spearman = -0.11 (p = 0.45), n = 47. La humedad relativa media de esas fechas va de 74.2 % a 89.0 %.",
  "interpretacion": "En estos datos no se detecta una asociación entre la humedad relativa del día y la infestación promedio: el signo es ligeramente negativo, pero es débil y no es estadísticamente distinguible de cero.",
  "limitacion": "Mismas limitaciones de la Figura 2: mismo día, una sola estación, un ciclo y fechas con pocas evaluaciones. Con el promedio de 7 días, rho = 0.02 (p = 0.89, n = 45); solo con n >= 10, rho = 0.10. Una correlación no demuestra causa.",
  "decision": "No incluir la humedad relativa como variable de alerta por ahora. Conservarla en el dataset para revisar la relación cuando existan más campañas."
 },
 "Figura 4 (complementaria)": {
  "titulo": "Mapa de calor módulo × mes del % promedio de hojas infestadas",
  "pregunta": "¿En qué módulos y meses se concentra el mayor % de hojas infestadas?",
  "variables": "dim_modulo.modulo; evaluacion_fitosanitaria.fecha (mes) y aranita_hojas_infestadas (%); filtro ACEPTADO",
  "unidad": "Una evaluación fitosanitaria; n = 1,462, agrupadas por módulo × mes (las celdas con menos de 5 evaluaciones no se pintan).",
  "metrica": "Promedio del % de hojas infestadas por celda módulo × mes.",
  "visualizacion": "Matriz de calor: se comparan dos dimensiones categóricas (módulo y mes) con una medida numérica. Cada celda muestra el valor y su n.",
  "resultado": "El módulo 6 pasa de 0.6 % (mayo) a 27.4 % (agosto). Los módulos 4 y 5 superan 15 % de junio a agosto (módulo 4: 15.9, 19.4 y 19.7 %; módulo 5: 18.5, 17.4 y 23.7 %). Los módulos 2 y 3 no pasan de 12.9 % en ningún mes.",
  "interpretacion": "En los registros disponibles, la infestación se concentra en los módulos 4 y 5 desde junio y en el módulo 6 en agosto, mientras que los módulos 2 y 3 se mantienen por debajo de 13 %. Que el módulo 6 solo se eleve en agosto sugiere un aumento más tardío, no una infestación constante.",
  "limitacion": "El módulo 1 solo tiene 9 evaluaciones válidas y no se pinta. El n por celda va de 17 a 89. No se dispone de información sobre aplicaciones de control ni sobre diferencias de edad o variedad entre módulos; una baja puede deberse a un tratamiento. El promedio está influido por pocas válvulas con valores altos.",
  "decision": "Priorizar la revisión en campo de los módulos 4, 5 y 6 y definir con el Ing. Kiyoshi los umbrales del semáforo por módulo antes de fijarlos."
 }
}
CAMPOS = [["pregunta", "Pregunta analítica"], ["variables", "Variables (campos reales)"], ["unidad", "Unidad de análisis"], ["metrica", "Métrica / operación estadística"], ["visualizacion", "Visualización y justificación"], ["resultado", "Resultado observado (cifras)"], ["interpretacion", "Interpretación"], ["limitacion", "Limitación"], ["decision", "Decisión que podría evaluarse"]]
ARCH = "S12_AgroDataIC_v1.ipynb · aranita_roja_bd_v1.sqlite (v1) · aranita_roja_analitico_v1.csv"
PNG = {'Figura 1': 'figura_01_serie_temporal_infestacion.png', 'Figura 2': 'figura_02_dispersion_temperatura_infestacion.png',
       'Figura 3': 'figura_03_dispersion_humedad_infestacion.png', 'Figura 4 (complementaria)': 'figura_04_mapa_calor_modulo_mes.png'}

In [ ]:
# Tablas de respaldo: permiten que cualquiera compruebe los números de cada figura.
mensual.reset_index().rename(columns={'mes': 'mes_numero'}).to_csv(os.path.join(RAIZ, 'analisis', 'tabla_01_mensual.csv'), index=False, encoding='utf-8-sig')
por_fecha.assign(fecha=por_fecha['fecha'].dt.strftime('%Y-%m-%d')).to_csv(os.path.join(RAIZ, 'analisis', 'tabla_02_por_fecha.csv'), index=False, encoding='utf-8-sig')
pv.round(2).to_csv(os.path.join(RAIZ, 'analisis', 'tabla_03_modulo_mes_promedio.csv'), encoding='utf-8-sig')
pn.to_csv(os.path.join(RAIZ, 'analisis', 'tabla_04_modulo_mes_n.csv'), encoding='utf-8-sig')

# Matriz de interpretación (una fila por figura).
filas = []
for fig_id, f in FICHAS.items():
    fila = {'Figura': fig_id, 'Título': f['titulo']}
    for k, lab in CAMPOS:
        fila[lab] = f[k]
    fila['Archivo y versión'] = PNG[fig_id] + ' · ' + ARCH
    filas.append(fila)
matriz = pd.DataFrame(filas)
display(matriz[['Figura', 'Pregunta analítica', 'Resultado observado (cifras)']])
matriz.to_csv(os.path.join(RAIZ, 'matriz_interpretacion', 'matriz_interpretacion.csv'), index=False, encoding='utf-8-sig')
try:
    ruta_xlsx = os.path.join(RAIZ, 'matriz_interpretacion', 'matriz_interpretacion.xlsx')
    matriz.to_excel(ruta_xlsx, index=False, sheet_name='Matriz S12')
    # Formato legible: columnas anchas, texto ajustado, encabezado resaltado.
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    wb = load_workbook(ruta_xlsx); ws = wb.active
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 16 if col[0].column_letter in 'AB' else 42
        for cell in col:
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for cell in ws[1]:
        cell.font = Font(bold=True, color='FFFFFF'); cell.fill = PatternFill('solid', fgColor='0D5192')
    ws.freeze_panes = 'C2'
    wb.save(ruta_xlsx)
except Exception as e:
    print('No se pudo crear el .xlsx (se conserva el .csv):', e)

# Trazabilidad de los datos (formato sugerido en la diapositiva "Qué debemos entregar").
traza = '''Dataset: aranita_roja_bd_v1.sqlite (tablas clima_diario y evaluacion_fitosanitaria) -> aranita_roja_analitico_v1.csv
Fuente: Fundo Loma Verde S.A.C. (estación meteorológica y monitoreo fitosanitario de campo)
Versión: v1 (Semana 8)
Periodo: clima 2026-03-01 a 2026-08-30; evaluaciones 2026-03-02 a 2026-08-28
Transformaciones: ver notebook S12_AgroDataIC_v1.ipynb (JOIN clima + evaluación + módulo + turno; filtro estado_calidad_evaluacion = ACEPTADO)
Cálculos principales: promedio y mediana mensual del % de hojas infestadas; promedio por fecha; correlación de Spearman; promedio por módulo x mes
Limitaciones: un solo ciclo de 6 meses; rango térmico 20.4-23.2 C; Tbase=10.34 C y K=163.93 GDA son supuestos de literatura sin validar en campo
'''
with open(os.path.join(RAIZ, 'dataset_y_fuentes', 'fuente_o_referencia.txt'), 'w', encoding='utf-8') as fh:
    fh.write(traza)
print(traza)

In [ ]:
# Lista de cotejo de la docente (diapositiva "¿Nuestras visualizaciones están listas?"): lo que el código ya verificó.
control = [
 ('¿Se utilizó el dataset validado y documentado?', 'Sí: solo evaluaciones ACEPTADO de la base v1 (assert 1,462 filas).'),
 ('¿La unidad de análisis está definida?', 'Sí: evaluación (Figs. 1 y 4) y fecha de evaluación (Figs. 2 y 3).'),
 ('¿Cada gráfico responde una pregunta del proyecto?', 'Sí: ver la ficha de cada figura.'),
 ('¿La métrica y la agregación son correctas?', 'Sí: promedio sobre evaluaciones, no promedio de promedios; asserts con los valores esperados.'),
 ('¿Título, unidades y periodo en cada gráfico?', 'Sí: título, subtítulo con periodo y n, ejes con unidades.'),
 ('¿Los valores coinciden con las tablas de resultados?', 'Sí: carpeta analisis/ (tablas 01 a 04 y sensibilidad).'),
 ('¿La interpretación incluye cifras verificables?', 'Sí: las cifras principales de cada ficha tienen su assert.'),
 ('¿Se reconocen las limitaciones?', 'Sí: ficha de cada figura.'),
 ('¿La decisión es proporcional a la evidencia?', 'Sí: son revisiones y validaciones, no órdenes de intervención.'),
 ('¿Se puede reproducir desde el archivo entregado?', 'Sí: este notebook + aranita_roja_bd_v1.sqlite.'),
]
display(pd.DataFrame(control, columns=['Control', 'Estado']))

In [ ]:
# Descarga (Colab): empaqueta la carpeta de entrega en un .zip.
zip_path = shutil.make_archive('Semana_12_Visualizacion', 'zip', RAIZ)
print('Paquete creado:', zip_path)
if EN_COLAB:
    files.download(zip_path)

## Para exponer en 5 minutos (preguntas de sustentación de la docente)
Si se expone en clase, la docente no pide la ficha de cada gráfico. Estas son sus cinco preguntas, con las respuestas del proyecto.

**1. ¿Por qué eligieron este tipo de gráfico?** Porque la pregunta lo pide: cómo cambia la infestación en el tiempo (líneas), si dos variables cuantitativas cambian juntas (dispersión) y dónde y cuándo se concentra (matriz de calor, dos categorías y una medida). No se eligió por apariencia.

**2. ¿Qué representa cada fila de su dataset?** Una evaluación fitosanitaria: una fecha, un módulo, una válvula y un turno. En las dispersiones, cada punto es una fecha de evaluación, porque el clima es diario y no por evaluación.

**3. ¿Cómo calcularon la métrica?** Promedio del % de hojas infestadas sobre las evaluaciones aceptadas (por mes, por fecha o por módulo y mes). Después se verificó con asserts contra las tablas de la carpeta `analisis/`.

**4. ¿Qué resultado es el más relevante?** La infestación promedio sube de 3.0 % en marzo a 16.4 % en agosto, y los módulos 4 y 5 superan 15 % desde junio, mientras el módulo 6 se dispara en agosto (27.4 %). Además, la temperatura y la humedad diarias no muestran asociación con la infestación en estos datos.

**5. ¿Qué decisión puede informar y qué información falta?** Informa a qué módulos mirar primero y por qué el semáforo no debe apoyarse solo en la temperatura diaria. Falta: más campañas, un rango térmico más amplio, información sobre aplicaciones de control, y validar en campo Tbase = 10.34 °C y K = 163.93 GDA con el Ing. Kiyoshi.

**Pregunta probable del jurado:** *¿no encontraron relación con la temperatura, entonces para qué el grado-día?* Respuesta honesta: en 6 meses la temperatura casi no varía (20.4 a 23.2 °C), así que este conjunto no sirve para probar ni descartar el efecto térmico. El grado-día queda como contexto y como hipótesis a validar con más campañas; la alerta de hoy se apoya en la infestación observada en campo.